In [ ]:
#| default_exp learner

# Load deps

In [ ]:
# !pip install -q torcheval fastprogress
# # or
# !uv add torcheval fastprogress

In [ ]:
#|export
import math,torch,matplotlib.pyplot as plt
from collections.abc import Mapping
from operator import attrgetter
from functools import partial
from copy import copy
from torch import optim
import torch.nn.functional as F
from torcheval.metrics import MulticlassAccuracy,Mean
from fastprogress import progress_bar,master_bar

In [ ]:
# # if src modules imported
# from google.colab import drive
# drive.mount('/content/drive')
# import sys
# from pathlib import Path
# app_path = '/content/drive/MyDrive/Projects/miniSD'
# sys.path.append(app_path)

In [ ]:
#|export
from src.conv import def_device, to_device
from src.datasets import inplace, collate_dict
from src.utils import store_attr

In [ ]:
import matplotlib as mpl
import torchvision.transforms.functional as TF
from contextlib import contextmanager
from torch import nn,tensor
from datasets import load_dataset

# Config

In [ ]:
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray'
dataset_name = "fashion_mnist"

# Learner

In [ ]:
x,y = 'image','label'
dsd = load_dataset(dataset_name)

In [ ]:
@inplace
def transformi(b): b[x] = [torch.flatten(TF.to_tensor(o)) for o in b[x]]
tds = dsd.with_transform(transformi)

In [ ]:
#|export --> training
from torch.utils.data import DataLoader
def get_dls(train_ds, valid_ds, bs, collate_fn, min_bs = 8, **kwargs):
    last_train_batch_len = len(train_ds) % bs
    train_dl = DataLoader(
        train_ds,
        bs,
        shuffle=True,
        drop_last=(last_train_batch_len > 0) and (last_train_batch_len < min_bs),
        collate_fn=collate_fn,
        **kwargs
        )
    valid_dl = DataLoader(
        valid_ds,
        bs,
        shuffle=False,
        drop_last=False,
        collate_fn=collate_fn,
        **kwargs
        )
    return train_dl, valid_dl

In [ ]:
#|export --> datasets
class DataLoaders:
    def __init__(self, *dls): self.train,self.valid = dls[:2]
    @classmethod
    def from_dd(cls, dd, batch_size, as_tuple=True, **kwargs):
        f = collate_dict(dd['train'])
        return cls(*get_dls(*dd.values(), bs=batch_size, collate_fn=f, **kwargs))
        # TODO: we don't put anything to device here. Why??

In [ ]:
bs = 1024
dls = DataLoaders.from_dd(tds, bs, num_workers=2)
dt = dls.train
xb,yb = next(iter(dt))
print(xb.shape,yb[:10])

In [ ]:
class Learner:
    def __init__(self, model, dls, loss_func, lr, opt_func=optim.SGD, device = def_device):
        store_attr()

    def one_batch(self):
        self.xb,self.yb = to_device(self.batch)
        self.preds = self.model(self.xb)
        self.loss = self.loss_func(self.preds, self.yb)
        if self.model.training:
            self.loss.backward()
            self.opt.step()
            self.opt.zero_grad()
        with torch.no_grad(): self.calc_stats()

    def calc_stats(self):
        acc = (self.preds.argmax(dim=1)==self.yb).float().sum()
        self.accs.append(acc)
        n = len(self.xb)
        self.losses.append(self.loss*n)
        self.ns.append(n)

    def one_epoch(self, train):
        self.model.training = train
        dl = self.dls.train if train else self.dls.valid
        for self.num,self.batch in enumerate(dl): self.one_batch()
        n = sum(self.ns)
        print(f"epoch: {self.epoch}, training: {self.model.training},\
                loss: {sum(self.losses).item()/n}, accuracy: {sum(self.accs).item()/n}")

    def fit(self, n_epochs):
        self.accs,self.losses,self.ns = [],[],[]
        self.model.to(self.device)
        self.opt = self.opt_func(self.model.parameters(), self.lr)
        self.n_epochs = n_epochs
        for self.epoch in range(n_epochs):
            self.one_epoch(True)
            with torch.no_grad(): self.one_epoch(False)

In [ ]:
# m,nh = 28*28,50
# model = nn.Sequential(nn.Linear(m,nh), nn.ReLU(), nn.Linear(nh,10))
# learn = Learner(model, dls, F.cross_entropy, lr=0.2)
# learn.fit(1)

# Adding Basic Callbacks

In [ ]:
#|export
class CancelFitException(Exception): pass
class CancelBatchException(Exception): pass
class CancelEpochException(Exception): pass

class Callback: order = 0

def run_cbs(cbs, method_nm, learn=None):
    for cb in sorted(cbs, key=attrgetter('order')):
        method = getattr(cb, method_nm, None)
        if method is not None: method(learn)

In [ ]:
class CompletionCB(Callback):
    def before_fit(self, learn): self.count = 0
    def after_batch(self, learn): self.count += 1
    def after_fit(self, learn): print(f'Completed {self.count} batches')

# cbs = [CompletionCB()]
# run_cbs(cbs, 'before_fit')
# run_cbs(cbs, 'after_batch')
# run_cbs(cbs, 'after_fit')

In [ ]:
class Learner():
    def __init__(self, model, dls, loss_func, lr, cbs, opt_func=optim.SGD):
        store_attr()
    # in this version we don't send the model and the batch to device.
    # they are both on CPU. we'll define a CB to do this below.

    def one_batch(self):
        self.preds = self.model(self.batch[0])
        self.loss = self.loss_func(self.preds, self.batch[1])
        if self.model.training:
            self.loss.backward()
            self.opt.step()
            self.opt.zero_grad()

    def one_epoch(self, train):
        self.model.train(train)
        self.dl = self.dls.train if train else self.dls.valid
        try:
            self.callback('before_epoch')
            for self.iter,self.batch in enumerate(self.dl):
                try:
                    self.callback('before_batch')
                    self.one_batch()
                    self.callback('after_batch')
                except CancelBatchException: pass
            self.callback('after_epoch')
        except CancelEpochException: pass

    def fit(self, n_epochs):
        self.n_epochs = n_epochs
        self.epochs = range(n_epochs)
        self.opt = self.opt_func(self.model.parameters(), self.lr)
        try:
            self.callback('before_fit')
            for self.epoch in self.epochs:
                self.one_epoch(True)
                self.one_epoch(False)
            self.callback('after_fit')
        except CancelFitException: pass

    def callback(self, method_nm): run_cbs(self.cbs, method_nm, self)

In [ ]:
m,nh = 28*28,50
def get_model(): return nn.Sequential(nn.Linear(m,nh), nn.ReLU(), nn.Linear(nh,10))

# model = get_model()
# learn = Learner(model, dls, F.cross_entropy, lr=0.2, cbs=[CompletionCB()])
# learn.fit(1)
# # print(next(model.parameters()).device)

In [ ]:
#| export
class SingleBatchCB(Callback):
    order = 1
    def after_batch(self, learn):
      raise CancelEpochException() # single batch executed
      # raise CancelBatchException() # all batches executed
      # raise CancelFitException() # no batch executed

In [ ]:
# learn = Learner(get_model(), dls, F.cross_entropy, lr=0.2, cbs=[SingleBatchCB(), CompletionCB()])
# learn.fit(1)

# # why 2 batches are done (1 for train and 1 for eval??) in this case
# # but when we set CancelBatchException, 69 batches are completes?
# # sholdn't the number of batches be even at least? (train + eval)?
# # No. run and see below
# print(len(dls.train))
# print(len(dls.valid))

# Metrics

In [ ]:
class Metric:
    def __init__(self): self.reset()
    def reset(self): self.vals,self.ns = [],[]
    def add(self, inp, targ=None, n=1):
        self.last = self.calc(inp, targ)
        self.vals.append(self.last)
        self.ns.append(n)
    @property
    def value(self):
        ns = tensor(self.ns)
        return (tensor(self.vals)*ns).sum()/ns.sum()
    def calc(self, inps, targs): return inps

class Accuracy(Metric):
    def calc(self, inps, targs): return (inps==targs).float().mean()

acc = Accuracy()
acc.add(tensor([0, 1, 2, 0, 1, 2]), tensor([0, 1, 1, 2, 1, 0]))
acc.add(tensor([1, 1, 2, 0, 1]), tensor([0, 1, 1, 2, 1]))

#TODO: this should be the list of batch lenghts not all ones.
print(acc.ns)
print(acc.value)
print("===============")
loss = Metric()
loss.add(0.6, n=32)
loss.add(0.9, n=2)
print(loss.value, round((0.6*32+0.9*2)/(32+2), 2))
print("===============")

In [ ]:
metric = MulticlassAccuracy() # from torcheval.metrics
metric.update(tensor([0, 2, 1, 3]), tensor([0, 1, 2, 3]))
metric.update(tensor([0, 2, 1, 3]), tensor([0, 1, 2, 3]))
print(metric.compute())

# TODO: this is how the number of inputs should work!!
# fix the previous TODO
print(metric.num_total)
metric.reset()
print(metric.compute())

# Adding metric as callbacks

In [ ]:
#|export
def to_cpu(x):
    if isinstance(x, Mapping): return {k:to_cpu(v) for k,v in x.items()}
    if isinstance(x, list): return [to_cpu(o) for o in x]
    if isinstance(x, tuple): return tuple(to_cpu(list(x)))
    res = x.detach().cpu()
    return res.float() if res.dtype==torch.float16 else res

class MetricsCB(Callback):
    def __init__(self, *ms, **metrics):
        for o in ms: metrics[type(o).__name__] = o
        self.metrics = metrics
        self.all_metrics = copy(metrics)
        self.all_metrics['loss'] = self.loss = Mean()

    def _log(self, d): print(d)
    # we can change this later to another printing function
    # that's why we didn't just replace this with a simple print call
    def before_fit(self, learn): learn.metrics = self
    def before_epoch(self, learn): [o.reset() for o in self.all_metrics.values()]

    def after_epoch(self, learn):
        log = {k:f'{v.compute():.3f}' for k,v in self.all_metrics.items()}
        log['epoch'] = learn.epoch
        log['train'] = 'train' if learn.model.training else 'eval'
        self._log(log)

    def after_batch(self, learn):
        x,y,*_ = learn.batch
        x,y = to_cpu((x,y))
        for m in self.metrics.values(): m.update(to_cpu(learn.preds), y)
        # loss update is a bit different from other metrics
        self.loss.update(to_cpu(learn.loss), weight=len(x))

In [ ]:
#|export
class DeviceCB(Callback):
    def __init__(self, device=def_device): store_attr()
    def before_fit(self, learn):
        if hasattr(learn.model, 'to'): learn.model.to(self.device)
    def before_batch(self, learn): learn.batch = to_device(learn.batch, device=self.device)

In [ ]:
# model = get_model()
# metrics = MetricsCB(accuracy=MulticlassAccuracy())
# learn = Learner(model, dls, F.cross_entropy, lr=0.2, cbs=[DeviceCB(), metrics])
# learn.fit(1)

In [ ]:
# model = get_model()
# metrics = MetricsCB(MulticlassAccuracy())
# learn = Learner(model, dls, F.cross_entropy, lr=0.2, cbs=[DeviceCB(), metrics])
# learn.fit(1)

# Flexible learner v1

In [ ]:
class Learner():
  def __init__(
      self, model, dls,
      loss_func=F.mse_loss,
      lr=0.1,
      cbs=[],
      opt_func=optim.SGD
  ):store_attr()

  @contextmanager
  def cb_ctx(self, nm):
      try:
          self.callback(f'before_{nm}')
          yield
          # anything yielded here can be assingned to x with `with ... as x` below
          self.callback(f'after_{nm}')
      except globals()[f'Cancel{nm.title()}Exception']: pass
      finally: self.callback(f'cleanup_{nm}')

  def one_epoch(self, train):
      self.model.train(train)
      self.dl = self.dls.train if train else self.dls.valid
      with self.cb_ctx('epoch'):
          for self.iter,self.batch in enumerate(self.dl):
              with self.cb_ctx('batch'):
                  self.predict()
                  self.get_loss()
                  if self.training:
                      self.backward()
                      self.step()
                      self.zero_grad()

  def fit(self, n_epochs=1, train=True, valid=True, cbs=[], lr=None):
      for cb in cbs: self.cbs.append(cb)
      try:
          self.n_epochs = n_epochs
          self.epochs = range(n_epochs)
          self.opt = self.opt_func(self.model.parameters(), self.lr if lr is None else lr)
          with self.cb_ctx('fit'):
              for self.epoch in self.epochs:
                  if train: self.one_epoch(True)
                  if valid: torch.no_grad()(self.one_epoch)(False)
      finally:
          for cb in cbs: self.cbs.remove(cb)

  def __getattr__(self, name):
    # this is called if the methods is not defined regularly.
    # So, for example, another class can inherit from this class and define a special `predict`
    # Or, define a different callback with a different predict method
    if name in ('predict','get_loss','backward','step','zero_grad'): return partial(self.callback, name)
    raise AttributeError(name)

  def callback(self, method_nm): run_cbs(self.cbs, method_nm, self)

  @property
  def training(self): return self.model.training

In [ ]:
#|export
class TrainCB(Callback):
  # TODO: what happens if another CB has a method with a common name with this CB???
  # They would be chained based on CB order but is this the desired behavior??
  def __init__(self, n_inp=1): self.n_inp = n_inp
  # `n_inp` allows us to train models with more than one input or output.
  def predict(self, learn): learn.preds = learn.model(*learn.batch[:self.n_inp])
  def get_loss(self, learn): learn.loss = learn.loss_func(learn.preds, *learn.batch[self.n_inp:])
  def backward(self, learn): learn.loss.backward()
  def step(self, learn): learn.opt.step()
  def zero_grad(self, learn): learn.opt.zero_grad()

In [ ]:
#|export
class ProgressCB(Callback):
    order = MetricsCB.order+1
    def __init__(self, plot=False): self.plot = plot
    def before_fit(self, learn):
        learn.epochs = self.mbar = master_bar(learn.epochs)
        self.first = True
        # this is why we define the metric's log as a separate module
        # in order to be able to inject a different log using a CB
        if hasattr(learn, 'metrics'): learn.metrics._log = self._log
        # shouldn't we revert all changes of this kind in an `after_fit` method?
        # Not needed. the `MetricsCB` resets this in its own `before_fit` method
        self.losses = []
        self.val_losses = []

    def _log(self, d):
        if self.first:
            self.mbar.write(list(d), table=True)
            self.first = False
        self.mbar.write(list(d.values()), table=True)

    def before_epoch(self, learn):
      learn.dl = progress_bar(learn.dl, leave=False, parent=self.mbar)
      # this state is also reset in the Learner's `one_epoch` method
    def after_batch(self, learn):
        learn.dl.comment = f'{learn.loss:.3f}'
        if self.plot and hasattr(learn, 'metrics') and learn.training:
            self.losses.append(learn.loss.item())
            if self.val_losses:
              self.mbar.update_graph(
                [
                    [range(len(self.losses)), self.losses],
                    [list(map(lambda x: (x+1)*len(learn.dls.train), range(learn.epoch))), self.val_losses]
                ]
            )
            else: self.mbar.update_graph([[range(len(self.losses)), self.losses]])

    def after_epoch(self, learn):
        if not learn.training:
            if self.plot and hasattr(learn, 'metrics'):
                self.val_losses.append(learn.metrics.all_metrics['loss'].compute())
                self.mbar.update_graph(
                    [
                        [range(len(self.losses)), self.losses],
                        [list(map(lambda x: (x+1)*len(learn.dls.train), range(learn.epoch+1))), self.val_losses]
                    ]
                )

In [ ]:
# model = get_model()
# cbs = [
#     TrainCB(),
#     DeviceCB(),
#     MetricsCB(accuracy=MulticlassAccuracy()),
#     ProgressCB(plot=True)
# ]
# learn = Learner(model, dls, F.cross_entropy, lr=0.2, cbs=cbs)
# learn.fit(2)

# print(learn.cbs[-1].val_losses)
# # valid loss has as many points as the number epochs.
# # not so informative.
# # TODO: change the eval_strategy to steps so that
# # - it starts calculating valid loss earlier and
# # - does that more often

## Fixing the context manager
- what happens if we want to raise an exception before the `yield` here:
```python
def cb_ctx(self, nm):
  try:
      self.callback(f'before_{nm}')
      yield
      self.callback(f'after_{nm}')
```

In [ ]:
class VerboseMetricCB(MetricsCB):
  def after_batch(self, learn):
    x,y,*_ = learn.batch
    x,y = to_cpu((x,y))
    for m in self.metrics.values(): m.update(to_cpu(learn.preds), y)
    # loss update is a bit different from other metrics
    self.loss.update(to_cpu(learn.loss), weight=len(x))
    self._log(f"batch loss: {learn.loss}")

class ZeroBatchCB(Callback):
  order = 2
  def before_batch(self, learn):
    # Raises the error
    raise CancelBatchException()
  def after_batch(self, learn):
    # No effect. caught by try block
    raise CancelBatchException()


# model = get_model()
# cbs = [
#     ZeroBatchCB(),
#     TrainCB(),
#     DeviceCB(),
#     VerboseMetricCB(accuracy=MulticlassAccuracy())
# ]
# learn = Learner(model, dls, F.cross_entropy, lr=0.2, cbs=cbs)
# learn.fit(1)

# # RuntimeError: generator didn't yield

In [ ]:
#|export
class with_cbs:
    def __init__(self, nm): self.nm = nm
    def __call__(self, f):
        def _f(o, *args, **kwargs):
            try:
                o.callback(f'before_{self.nm}')
                f(o, *args, **kwargs)
                o.callback(f'after_{self.nm}')
            except globals()[f'Cancel{self.nm.title()}Exception']: pass
            finally: o.callback(f'cleanup_{self.nm}')
        return _f

In [ ]:
#|export
class Learner():
  def __init__(
      self, model, dls,
      loss_func=F.mse_loss,
      lr=0.1,
      cbs=[],
      opt_func=optim.SGD
  ):store_attr()

  @with_cbs('batch')
  def _one_batch(self):
    self.predict()
    self.callback('after_predict')
    self.get_loss()
    self.callback('after_loss')
    if self.training:
      self.backward()
      self.callback('after_backward')
      self.step()
      self.callback('after_step')
      self.zero_grad()
  @with_cbs('epoch')
  def _one_epoch(self):
    for self.iter,self.batch in enumerate(self.dl): self._one_batch()
  def one_epoch(self, training):
    self.model.train(training)
    self.dl = self.dls.train if training else self.dls.valid
    self._one_epoch()
  @with_cbs('fit')
  def _fit(self, train, valid):
    for self.epoch in self.epochs:
      if train: self.one_epoch(True)
      if valid: torch.no_grad()(self.one_epoch)(False)

  def fit(self, n_epochs=1, train=True, valid=True, cbs=[], lr=None):
    for cb in cbs: self.cbs.append(cb)
    try:
      self.n_epochs = n_epochs
      self.epochs = range(n_epochs)
      self.opt = self.opt_func(self.model.parameters(), self.lr if lr is None else lr)
      self._fit(train, valid)
    finally:
      for cb in cbs: self.cbs.remove(cb)

  def __getattr__(self, name):
    if name in ('predict','get_loss','backward','step','zero_grad'):
      return partial(self.callback, name)
    raise AttributeError(name)

  def callback(self, method_nm): run_cbs(self.cbs, method_nm, self)

  @property
  def training(self): return self.model.training

In [ ]:
# # model = get_model()
# # metrics = MetricsCB(accuracy=MulticlassAccuracy())
# # cbs = [TrainCB(), DeviceCB(), metrics, ProgressCB(plot=True)]
# # learn = Learner(model, dls, F.cross_entropy, lr=0.2, cbs=cbs)
# # learn.fit(2)

# model = get_model()
# cbs = [
#     ZeroBatchCB(),
#     TrainCB(),
#     DeviceCB(),
#     VerboseMetricCB(accuracy=MulticlassAccuracy())
# ]
# learn = Learner(model, dls, F.cross_entropy, lr=0.2, cbs=cbs)
# learn.fit(1)

# MomentumLearner

In [ ]:
#|export
class TrainLearner(Learner):
  def predict(self): self.preds = self.model(self.batch[0])
  def get_loss(self): self.loss = self.loss_func(self.preds, self.batch[1])
  def backward(self): self.loss.backward()
  def step(self): self.opt.step()
  def zero_grad(self): self.opt.zero_grad()

class MomentumLearner(TrainLearner):
  def __init__(self, model, dls, loss_func, lr, cbs=[], opt_func=optim.SGD, mom=0.85):
    self.mom = mom
    super().__init__(model, dls, loss_func, lr, cbs, opt_func)

  def zero_grad(self):
    with torch.no_grad():
      for p in self.model.parameters(): p.grad *= self.mom

In [ ]:
metrics = MetricsCB(accuracy=MulticlassAccuracy())
cbs = [DeviceCB(), metrics, ProgressCB(plot=True)] # No TrainCB
learn = MomentumLearner(get_model(), dls, F.cross_entropy, lr=0.1, cbs=cbs)
learn.fit(2)

## Alternative implementation

In [ ]:
#|export
class MomentumTrainCB(TrainCB):
  def __init__(self, mom=0.85):
    self.mom = mom
    super().__init__()
  def zero_grad(self, learn):
    with torch.no_grad():
      for p in learn.model.parameters(): p.grad *= self.mom

In [ ]:
# model = get_model()
# metrics = MetricsCB(accuracy=MulticlassAccuracy())
# cbs = [MomentumTrainCB(), DeviceCB(), metrics, ProgressCB(plot=True)]
# learn = Learner(model, dls, F.cross_entropy, lr=0.2, cbs=cbs)
# learn.fit(2)

# LRFinder callback

In [ ]:
class LRFinderCB(Callback):
  def __init__(self, lr_mult=1.3): store_attr()
  def before_fit(self, learn):
    self.lrs,self.losses = [],[]
    self.min = math.inf

  def after_batch(self, learn):
    if not learn.training: raise CancelEpochException()
    self.lrs.append(learn.opt.param_groups[0]['lr'])
    loss = to_cpu(learn.loss)
    self.losses.append(loss)
    if loss < self.min: self.min = loss
    if loss > self.min*3: raise CancelFitException()
    for g in learn.opt.param_groups: g['lr'] *= self.lr_mult

lrfind = LRFinderCB()
cbs = [DeviceCB(), lrfind]
learn = MomentumLearner(get_model(), dls, F.cross_entropy, lr=1e-4, cbs=cbs)
learn.fit(1)
plt.plot(lrfind.lrs, lrfind.losses)
plt.xscale('log')

## Using pytorch exponential lr scheduler

In [ ]:
#|export
from torch.optim.lr_scheduler import ExponentialLR

class LRFinderCB(Callback):
  def __init__(self, gamma=1.3, max_mult=3): store_attr()
  def before_fit(self, learn):
    self.sched = ExponentialLR(learn.opt, self.gamma)
    self.lrs,self.losses = [],[]
    self.min = math.inf

  def after_batch(self, learn):
    if not learn.training: raise CancelEpochException()
    self.lrs.append(learn.opt.param_groups[0]['lr'])
    loss = to_cpu(learn.loss)
    self.losses.append(loss)
    if loss < self.min: self.min = loss
    if math.isnan(loss) or (loss > self.min*self.max_mult):
      raise CancelFitException()
    self.sched.step()

  def cleanup_fit(self, learn):
    plt.plot(self.lrs, self.losses)
    plt.xscale('log')

In [ ]:
cbs = [DeviceCB()]
learn = MomentumLearner(get_model(), dls, F.cross_entropy, lr=1e-5, cbs=cbs)
learn.fit(3, cbs=[LRFinderCB()])

## Integrate it into the learner class

In [ ]:
#|export
def lr_find(self:Learner, gamma=1.3, max_mult=3, start_lr=1e-5, max_epochs=10):
    self.fit(max_epochs, lr=start_lr, cbs=[LRFinderCB(gamma=gamma, max_mult=max_mult)])

Learner.lr_find = lr_find

In [ ]:
MomentumLearner(get_model(), dls, F.cross_entropy, 0.1, cbs=cbs).lr_find()